## Row count check
Compares every table in `workspace.iaa` with the row counts of the Kaggle Week 1 delivery. Stops the run if any table differs, and logs each run to `workspace.iaa.dq_row_count_log`.

In [0]:
CREATE OR REPLACE TEMP VIEW row_count_check AS
WITH expected (table_name, expected_rows) AS (
  VALUES ('addresses', 575983), ('claims', 163150), ('customers', 545240),
         ('policies_buildings', 707726), ('policies_contents', 721617), ('policies_motor', 979968),
         ('sample_submission', 626639), ('test_buildings', 184155), ('test_contents', 187498),
         ('test_motor', 254986), ('weather', 2879915)
),
actual AS (
            SELECT 'addresses'          AS table_name, COUNT(*) AS actual_rows FROM workspace.iaa.addresses
  UNION ALL SELECT 'claims',             COUNT(*) FROM workspace.iaa.claims
  UNION ALL SELECT 'customers',          COUNT(*) FROM workspace.iaa.customers
  UNION ALL SELECT 'policies_buildings', COUNT(*) FROM workspace.iaa.policies_buildings
  UNION ALL SELECT 'policies_contents',  COUNT(*) FROM workspace.iaa.policies_contents
  UNION ALL SELECT 'policies_motor',     COUNT(*) FROM workspace.iaa.policies_motor
  UNION ALL SELECT 'sample_submission',  COUNT(*) FROM workspace.iaa.sample_submission
  UNION ALL SELECT 'test_buildings',     COUNT(*) FROM workspace.iaa.test_buildings
  UNION ALL SELECT 'test_contents',      COUNT(*) FROM workspace.iaa.test_contents
  UNION ALL SELECT 'test_motor',         COUNT(*) FROM workspace.iaa.test_motor
  UNION ALL SELECT 'weather',            COUNT(*) FROM workspace.iaa.weather
)
SELECT e.table_name, e.expected_rows, a.actual_rows,
       a.actual_rows - e.expected_rows AS difference,
       CASE WHEN a.actual_rows = e.expected_rows THEN 'PASS' ELSE 'FAIL' END AS status
FROM expected e
LEFT JOIN actual a USING (table_name);

SELECT * FROM row_count_check ORDER BY table_name;

In [0]:
CREATE TABLE IF NOT EXISTS workspace.iaa.dq_row_count_log (
  checked_at TIMESTAMP, table_name STRING, expected_rows BIGINT, actual_rows BIGINT, status STRING);

INSERT INTO workspace.iaa.dq_row_count_log
SELECT current_timestamp(), table_name, expected_rows, actual_rows, status FROM row_count_check;

In [0]:
%sql
DESCRIBE TABLE workspace.iaa.policies_motor